In [17]:
%pip install gym gymnasium "shimmy>=2.0" yfinance

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 26.3 MB/s  0:00:007.8 MB/s eta 0:00:01
  Created wheel for multitasking: filename=multitasking-0.0.12-py3-none-any.whl size=15635 sha256=2e1368908dcbb35e34c241d9a71eb56dad145049afb8adf532201aaaa05474f3
  Stored in directory: /home/dungvo/.cache/pip/wheels/cc/bd/6f/664d62c99327abeef7d86489e6631cbf45b56fbf7ef1d6ef00
Successfully built multitasking
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [yfinance]━━ 7/8 [yfinance]
Note: you may need to restart the kernel to use updated packages.


In [10]:
import numpy as np
import gymnasium as gym
from gymnasium import spaces
import argparse

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import SubprocVecEnv


# =========================
# Portfolio Environment
# =========================
class PortfolioEnv(gym.Env):
    def __init__(self, prices, window_size=60, initial_cash=100000):
        super().__init__()

        self.prices = prices
        self.returns = np.log(prices[1:] / prices[:-1])

        self.window_size = window_size
        self.n_assets = prices.shape[1]
        self.initial_cash = initial_cash

        self.action_space = spaces.Box(
            low=-5, high=5,
            shape=(self.n_assets,),
            dtype=np.float32
        )

        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf,
            shape=(self.n_assets, window_size),
            dtype=np.float32
        )

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
    
        self.t = self.window_size
        self.portfolio_value = self.initial_cash
        self.weights = np.ones(self.n_assets) / self.n_assets
    
        self.A = 0.0
        self.B = 0.0
    
        return self._get_state(), {}

    def step(self, action):
        weights = self._softmax(action)
    
        r_t = self.returns[self.t]
    
        # -------------------------
        # 1. Raw portfolio return
        # -------------------------
        portfolio_return = np.dot(weights, r_t)
    
        # -------------------------
        # 2. Transaction cost
        # -------------------------
        transaction_cost_rate = 0.001  # 0.1% per trade
    
        turnover = np.sum(np.abs(weights - self.weights))
        cost = transaction_cost_rate * turnover
    
        # subtract cost from return
        portfolio_return -= cost
    
        # -------------------------
        # 3. Update portfolio value
        # -------------------------
        self.portfolio_value *= np.exp(portfolio_return)
    
        # -------------------------
        # 4. Reward (DSR)
        # -------------------------
        reward = self._differential_sharpe(portfolio_return)
        # reward = np.clip(reward, -10.0, 10.0)
        self.t += 1
        terminated = self.t >= len(self.returns) - 1
        truncated = False
    
        self.weights = weights
        state = self._get_state()
    
        return state, reward, terminated, truncated, {}

    def _get_state(self):
        window = self.returns[self.t - self.window_size:self.t]
        return window.T.astype(np.float32)

    def _softmax(self, x):
        e_x = np.exp(x - np.max(x))
        return e_x / (np.sum(e_x) + 1e-8)

    def _differential_sharpe(self, R_t, eta=1/252):
        delta_A = R_t - self.A
        delta_B = R_t**2 - self.B

        numerator = self.B * delta_A - 0.5 * self.A * delta_B
        denominator = (self.B - self.A**2 + 1e-8) ** 1.5

        D_t = numerator / (denominator + 1e-8)

        self.A += eta * delta_A
        self.B += eta * delta_B

        return D_t


# =========================
# Vectorized Environment
# =========================
def make_env(prices, window_size, initial_cash):
    def _init():
        return PortfolioEnv(prices, window_size, initial_cash)
    return _init


def create_vector_env(prices, n_envs, window_size, initial_cash):
    return SubprocVecEnv([
        make_env(prices, window_size, initial_cash)
        for _ in range(n_envs)
    ])


# =========================
# Training
# =========================
def train_ppo(prices, args):
    env = create_vector_env(
        prices,
        n_envs=args.n_envs,
        window_size=args.window_size,
        initial_cash=args.initial_cash
    )

    model = PPO(
        "MlpPolicy",
        env,
        verbose=1,
        n_steps=args.n_steps,
        batch_size=args.batch_size,
        n_epochs=args.n_epochs,
        gamma=args.gamma,
        gae_lambda=args.gae_lambda,
        clip_range=args.clip_range,
        learning_rate=args.learning_rate,
        policy_kwargs=dict(
            net_arch=[args.hidden_size, args.hidden_size]
        )
    )

    model.learn(total_timesteps=args.total_timesteps)

    return model


# =========================
# Evaluation
# =========================
def evaluate(model, prices, args):
    env = PortfolioEnv(
        prices,
        window_size=args.window_size,
        initial_cash=args.initial_cash
    )

    state, _ = env.reset()   # unpack properly
    done = False

    portfolio_values = [env.portfolio_value]

    while not done:
        action, _ = model.predict(state, deterministic=True)

        state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        portfolio_values.append(env.portfolio_value)

    return portfolio_values


# =========================
# Argument Parser
# =========================
def parse_args():
    parser = argparse.ArgumentParser()

    # Environment
    parser.add_argument("--window_size", type=int, default=60)
    parser.add_argument("--initial_cash", type=float, default=100000)

    parser.add_argument("--tickers", type=list, defaults=["XLK"])

    # PPO core
    parser.add_argument("--n_envs", type=int, default=10)
    parser.add_argument("--n_steps", type=int, default=756)
    parser.add_argument("--batch_size", type=int, default=1260)
    parser.add_argument("--n_epochs", type=int, default=16)

    # RL params
    parser.add_argument("--gamma", type=float, default=0.9)
    parser.add_argument("--gae_lambda", type=float, default=0.9)
    parser.add_argument("--clip_range", type=float, default=0.25)
    parser.add_argument("--learning_rate", type=float, default=3e-4)

    # Network
    parser.add_argument("--hidden_size", type=int, default=64)

    # Training
    parser.add_argument("--total_timesteps", type=int, default=7_500_000)

    return parser.parse_args()



In [14]:

default_args = {
    # Environment
    "window_size": 60,
    "initial_cash": 10000,
    "tickers": ["XLK"],
    # PPO core
    "n_envs": 10,
    "n_steps": 756,
    "batch_size": 4096,
    "n_epochs": 32,

    # RL params
    "gamma": 0.9,
    "gae_lambda": 0.9,
    "clip_range": 0.25,
    "learning_rate": 3e-4,

    # Network
    "hidden_size": 64,

    # Training
    "total_timesteps": 2500000
}
from types import SimpleNamespace

args = SimpleNamespace(**default_args)


from data import build_dataset

# Load data
prices_np, _, _, _ = build_dataset(
    tickers=args.tickers,
    start="2010-01-01",
    end="2024-01-01",
)
split = int(0.8 * len(prices_np))

train_prices = prices_np[:split]
test_prices  = prices_np[split:]
print("Loaded prices:", prices_np.shape)

model = train_ppo(train_prices,args)


In [16]:

# Evaluate
values = evaluate(model, test_prices, args)

print("Final Portfolio Value:", values[-1])

Final Portfolio Value: 9264.705951943033


Prices stats:
mean: 52.966602
min: 9.24528


In [31]:
import yfinance as yf